In [ ]:
!git clone https://github.com/archie-2006/fake-job-post-detection.git
%cd fake-job-post-detection

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn joblib

In [ ]:
!wget -O fake_job_postings.csv https://raw.githubusercontent.com/abbylmm/fake_job_posting/main/data/fake_job_postings.csv

In [ ]:
import os

os.makedirs('data/raw', exist_ok=True)
os.makedirs('models', exist_ok=True)

import shutil
shutil.move('fake_job_postings.csv', 'data/raw/fake_job_postings.csv')

In [ ]:
!python main.py

# Fake Job Posting Detection
## 1. Project Overview and Data Loading
This notebook documents the end-to-end pipeline for detecting fraudulent job postings. The dataset exhibits a severe class imbalance, which dictates our evaluation metrics and preprocessing strategy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from collections import Counter
import itertools
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

warnings.filterwarnings('ignore')

In [ ]:
# Load the raw dataset
df = pd.read_csv('data/raw/fake_job_postings.csv')
print(f"Dataset shape: {df.shape}")
print(f"Target distribution:\n{df['fraudulent'].value_counts(normalize=True)}")

## 2. Exploratory Data Analysis
### 2.1 Missing Values & Categorical Fraud Rate
Before modeling, we analyzed missing values, text length distributions, and categorical fraud rates to inform our imputation and feature engineering strategies. We also explored initial word frequencies to identify potential fraud triggers.

* Text columns with missing values will be filled with empty strings.
* Categorical columns with missing values will be explicitly assigned an 'Unknown' category.

In [ ]:
# Missing Value Analysis
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df)) * 100
print("Top Missing Value Percentages:")
print(missing_pct[missing_pct > 0].head())

# Categorical Fraud Rate Example
print("\nFraud Rate by Employment Type:")
print(df.groupby('employment_type')['fraudulent'].mean().sort_values(ascending=False))

### 2.2 Visualizing the Class Imbalance
Understanding the distribution of our target variable is critical. A severe imbalance dictates our entire modeling strategy, from how we split our training data (requiring stratification) to the metrics we must use to evaluate success (prioritizing F1-Score and PR-AUC over standard Accuracy).

In [ ]:
# Visualizing the Severe Class Imbalance
plt.figure(figsize=(7, 4))
ax = sns.countplot(data=df, x='fraudulent', palette=['#2ecc71', '#e74c3c'])
plt.title('Class Distribution: Genuine (0) vs Fraud (1)', fontsize=14)
plt.ylabel('Number of Postings', fontsize=12)

# Add exact counts on top of bars
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', 
        (p.get_x() + p.get_width() / 2., p.get_height()), 
        ha='center', va='bottom', fontsize=11)
plt.show()

### 2.3 Text Length Analysis
We hypothesize that fraudulent job postings might have different structural characteristics. Let's compare the word counts of the job descriptions between real and fake postings.

In [ ]:
# Calculate word count for descriptions
df['desc_word_count'] = df['description'].astype(str).str.split().str.len()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='fraudulent', y='desc_word_count', palette=['#2ecc71', '#e74c3c'])
plt.title('Distribution of Description Word Counts', fontsize=14)
plt.xlabel('0 = Genuine, 1 = Fraud', fontsize=12)
plt.ylabel('Word Count', fontsize=12)
plt.ylim(0, 800) # Cap outliers to make the boxplot readable
plt.show()

### 2.4 Categorical Risk Factors
Certain job categories attract more fraudulent activity. By plotting the fraud rate (percentage of fake jobs) across different categories, we can see which features will be most valuable for our classifier.

In [ ]:
# Calculate fraud rates for top departments
dept_counts = df['department'].value_counts()

# Only look at departments with at least 50 postings to avoid noise
valid_depts = dept_counts[dept_counts > 50].index
dept_fraud_rate = df[df['department'].isin(valid_depts)].groupby('department')['fraudulent'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=dept_fraud_rate.values * 100, y=dept_fraud_rate.index, palette='Reds_r')
plt.title('Top 10 Departments by Fraud Rate (%)', fontsize=14)
plt.xlabel('Percentage of Postings that are Fake', fontsize=12)
plt.ylabel('Department', fontsize=12)
plt.show()

### 2.5 Boolean Feature Analysis
Legitimate companies generally have established profiles, while fraudulent posters may skip optional steps to save time. Here, we analyze binary flags—such as the presence of a company logo or screening questions—to test if a lack of verification correlates with higher fraud rates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.barplot(data=df, x='has_company_logo', y='fraudulent', ax=axes, palette='Blues')
axes.set_title('Fraud Rate by Company Logo Presence')
axes.set_ylabel('Fraud Rate (%)')
axes.set_xticklabels(['No Logo (0)', 'Has Logo (1)'])

sns.barplot(data=df, x='has_questions', y='fraudulent', ax=axes, palette='Purples')
axes.set_title('Fraud Rate by Screening Questions')
axes.set_ylabel('')
axes.set_xticklabels(['No Questions (0)', 'Has Questions (1)'])

plt.tight_layout()
plt.show()

### 2.6 Word Frequency in Fraudulent Postings
Before relying on our models to extract feature weights, we can perform a foundational Bag-of-Words analysis. By filtering out common stop words, we isolate the most frequent vocabulary used strictly within fraudulent descriptions to establish our initial fraud-trigger hypothesis.

In [ ]:
# Isolate fake text
fake_text = df[df['fraudulent'] == 1]['description'].fillna('').str.lower()
fake_text = fake_text.str.replace(r'[^a-z\s]', '', regex=True) # Basic clean

# Count words
all_fake_words = list(itertools.chain.from_iterable(fake_text.str.split()))

# Filter out some basic stop words manually for the EDA plot
stop_words = {'and', 'the', 'to', 'of', 'in', 'a', 'for', 'is', 'with', 'on', 'this', 'you', 'are', 'we', 'be', 'that', 'as', 'it', 'or', 'your'}
filtered_words = [w for w in all_fake_words if w not in stop_words]

word_freq = Counter(filtered_words)
top_20 = word_freq.most_common(20)
words, counts = zip(*top_20)

plt.figure(figsize=(10, 6))
sns.barplot(x=list(counts), y=list(words), palette='Reds_r')
plt.title('Top 20 Most Frequent Words in Fraudulent Descriptions', fontsize=14)
plt.xlabel('Frequency', fontsize=12)
plt.show()

## 3. Preprocessing Pipeline
The preprocessing stage transforms raw text and categorical variables into a sparse matrix. To prevent data leakage, all transformations are fitted exclusively on the training set.

**Methodology:**
* **Text Cleaning:** Lowercasing, HTML tag removal, and whitespace normalization.
* **Train/Test Split:** Stratified 70/30 split to preserve the 5% fraud distribution.
* **TF-IDF Vectorization:** Capped at 5000 features with bigrams to control dimensionality.
* **One-Hot Encoding:** Applied to 5 categorical features, dropping original text.
* **Sparse Stacking:** Combined into a highly memory-efficient compressed sparse row (CSR) matrix.

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', '', text) 
    text = re.sub(r'[^a-z\s]', ' ', text) 
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Stratified Split
X = df.drop('fraudulent', axis=1)
y = df['fraudulent']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)

# Text Processing
text_cols = ['title', 'description', 'requirements', 'company_profile']
X_train_text = X_train[text_cols].fillna('').agg(' '.join, axis=1).apply(clean_text)
X_test_text = X_test[text_cols].fillna('').agg(' '.join, axis=1).apply(clean_text)

# TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.7, ngram_range=(1, 2), stop_words='english', dtype=np.float32)
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

# Categorical Encoding
cat_cols = ['location', 'department', 'employment_type', 'required_experience', 'industry']
cat_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.float32)
X_train_cat = cat_encoder.fit_transform(X_train[cat_cols].fillna('Unknown'))
X_test_cat = cat_encoder.transform(X_test[cat_cols].fillna('Unknown'))

# Final Sparse Matrix
X_train_processed = hstack([X_train_tfidf, X_train_cat]).tocsr()
X_test_processed = hstack([X_test_tfidf, X_test_cat]).tocsr()

train_sparsity = 100 * (1.0 - X_train_processed.nnz / (X_train_processed.shape * X_train_processed.shape))

print(f"Final Train Shape: {X_train_processed.shape}")
print(f"Final Test Shape:  {X_test_processed.shape}")
print(f"Matrix Sparsity:   {train_sparsity:.2f}%")

## 4. Machine Learning Modeling
### Import Libraries

In [ ]:
import joblib
import numpy as np
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV

### Load Data and Cross Validation Setup
*(Note: Loading pre-processed data from disk as outputted by main.py)*

In [ ]:
X_train = joblib.load('data/processed/X_train.pkl')
y_train = joblib.load('data/processed/y_train.pkl')

print("Shape:", X_train.shape)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### Naive Bayes

In [ ]:
nb = MultinomialNB(alpha=1.0)

f1_nb = cross_val_score(nb, X_train, y_train, cv=cv, scoring='f1')
pr_nb = cross_val_score(nb, X_train, y_train, cv=cv, scoring='average_precision')

print("NB F1:", f1_nb.mean())
print("NB PR-AUC:", pr_nb.mean())

nb.fit(X_train, y_train)
joblib.dump(nb, 'models/nb.pkl')

### Logistic Regression

In [ ]:
lr = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

f1_lr = cross_val_score(lr, X_train, y_train, cv=cv, scoring='f1')
pr_lr = cross_val_score(lr, X_train, y_train, cv=cv, scoring='average_precision')

print("LR F1:", f1_lr.mean())
print("LR PR-AUC:", pr_lr.mean())

lr.fit(X_train, y_train)
joblib.dump(lr, 'models/lr_l1.pkl')

### Hyperparameter Tuning

In [ ]:
param_grid = {
    'C': [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]
}

grid = GridSearchCV(
    LogisticRegression(
        penalty='l1',
        solver='liblinear',
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    ),
    param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Best C:", grid.best_params_)
print("Best F1:", grid.best_score_)

best_lr = grid.best_estimator_
joblib.dump(best_lr, 'models/lr_l1_best.pkl')

### Sparsity Check

In [ ]:
coefs = best_lr.coef_

total = len(coefs)
nonzero = np.sum(coefs != 0)

print("Total:", total)
print("Non-zero:", nonzero)
print("Sparsity %:", ((total - nonzero) / total) * 100)

### TF-IDF Word Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import joblib

# 1. Load the saved TF-IDF Vectorizer
try:
    tfidf = joblib.load('models/tfidf_vectorizer.pkl')
    feature_names = tfidf.get_feature_names_out()
    idf_weights = tfidf.idf_

    # 2. Sort the weights to find the extremes
    sorted_idx = np.argsort(idf_weights)
    top_common = [feature_names[i] for i in sorted_idx[:15]]
    top_rare = [feature_names[i] for i in sorted_idx[-15:]]

    print("Most common words (Lowest IDF):")
    print(", ".join(top_common))
    print("\nMost informative/rare words (Highest IDF):")
    print(", ".join(top_rare))
    print("-" * 50)

    # 3. Plot the distribution of IDF scores
    plt.figure(figsize=(10, 6))
    sns.histplot(idf_weights, bins=50, kde=True, color='#8E44AD')
    plt.title('Distribution of IDF Scores (Word Rarity)', fontsize=14, pad=15)
    plt.xlabel('IDF Score (Higher = More Rare & Informative)', fontsize=12)
    plt.ylabel('Number of Features (Words/N-grams)', fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # 4. Add some visual annotations
    plt.axvline(np.mean(idf_weights), color='red', linestyle='dashed', linewidth=2, label=f'Mean IDF: {np.mean(idf_weights):.2f}')
    plt.legend()
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print("Couldn't find 'models/tfidf_vectorizer.pkl'. Make sure you ran the main.py script successfully first")